# Lekcija 04 - Vzorec oblikovanja uporabe orodij

V tej lekciji boste spoznali vzorec oblikovanja **uporabe orodij** za AI agente z uporabo Microsoft Agent Framework (Python). Obravnavamo:

- Določanje funkcijskih orodij z dekoratorjem `@tool` in tipiziranimi parametri
- Zagotavljanje shem orodij, da model ve, kaj vsako orodje počne
- Nadzor izvajanja orodij z `approval_mode`
- Vračanje **strukturirane izhodne vrednosti** preko Pydantic modelov in `response_format`

Scenarij je **agent za rezervacijo potovanj**, ki lahko poišče destinacije, preveri razpoložljivost in pridobi informacije o letih.


## Namestitev


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Določanje orodij z dekoratorjem @tool

Dekorator `@tool` spremeni navadno Python funkcijo v orodje, ki ga lahko pokliče agent.
Ključne točke:

- **Dokumentacija** postane opis orodja, ki ga model vidi.
- **Tipne oznake** (vključno z `Annotated` z opisi) določajo shemo orodja.
- `approval_mode` nadzoruje, ali mora uporabnik odobriti vsak klic, preden se izvede.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Ustvarjanje agenta z več orodji

Posredujte vsem trem orodjem odjemalcu, da lahko model uporabi katero koli izmed njih za odgovor na vprašanje uporabnika.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturiran izhod z orodji

Z nastavitvijo `response_format` na Pydantic model, je agent prisiljen vrniti dobro tipiziran JSON objekt namesto prosto oblike besedila. To je koristno, kadar mora spodnja koda programatično uporabiti rezultat.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Vzorci odobritve orodij

Parameter `approval_mode` na `@tool` nadzoruje, ali klici orodja zahtevajo človeško odobritev pred izvajanjem:

| Način | Vedenje |
|---|---|
| `"never_require"` | Orodje se zažene samodejno — brez potrditve uporabnika. |
| `"always_require"` | Vsak klic mora odobriti uporabnik, preden se izvede. |

Uporabite `"always_require"` za orodja, ki imajo stranske učinke (npr. rezervacija leta, obračun kreditne kartice), da bo človek ostal v zanki.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Povzetek

V tej lekciji ste se naučili, kako:

1. **Določiti orodja** z uporabo dekoraterja `@tool` z tipiziranimi parametri in docstringi, ki služijo kot shema orodja.
2. **Sestaviti več orodij** tako, da jih agent lahko kliče zaporedoma za odgovarjanje na kompleksna vprašanja.
3. **Vriniti strukturiran izhod** tako, da kot `response_format` posredujete Pydantic model.
4. **Nadzorovati odobritev orodij** z `approval_mode`, da ohranite človeka v zanki pri občutljivih operacijah.

Ti vzorci tvorijo osnovo za gradnjo zanesljivih agentov, pripravljenih za produkcijo, ki lahko varno sodelujejo z zunanjimi sistemi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:  
Ta dokument je bil preveden z uporabo storitve za prevajanje z umetno inteligenco [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, upoštevajte, da lahko avtomatizirani prevodi vsebujejo napake ali netočnosti. Izvirni dokument v izvirnem jeziku velja za avtoritativni vir. Za pomembne informacije priporočamo strokovni človeški prevod. Za morebitne nesporazume ali napačne razlage, ki izhajajo iz uporabe tega prevoda, ne prevzemamo nobene odgovornosti.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
